In [ ]:
from langchain.chat_models import ChatOpenAI
from typing import Type
from langchain.tools import BaseTool
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage
from langchain.agents import initialize_agent, AgentType
from langchain.tools import DuckDuckGoSearchResults
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.document_loaders import WebBaseLoader
import requests
import os


llm = ChatOpenAI(
    temperature=0.1,
    model_name="gpt-4o-mini"
)

class SearchToolArgsSchema(BaseModel):
    query:str = Field(description="The query you will search for")

class DDGSearchTool(BaseTool):
    name:str = "DDGSearchTool"
    description:str = """
        Use this tool to research about given query.
    """

    args_schema: Type[SearchToolArgsSchema] = SearchToolArgsSchema

    def _run(self, query):
        ddg = DuckDuckGoSearchResults()
        return ddg.run(query)
    
class WikiSearchTool(BaseTool):
    name:str = "WikiSearchTool"
    description:str = """
        Use this tool to research about given query.
    """

    args_schema: Type[SearchToolArgsSchema] = SearchToolArgsSchema

    def _run(self, query):
        wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
        return wikipedia.run(query)

class SiteLoaderArgsSchema(BaseModel):
    url:str = Field(description="The url you will search for")    

class DDGSiteLoader(BaseTool):
    name:str = "DDGSiteLoader"
    description:str = """
        Use this tool to load web pages from given website url.
    """

    args_schema: Type[SiteLoaderArgsSchema] = SiteLoaderArgsSchema

    def _run(self, url):
        loader = WebBaseLoader(url)
        docs = loader.load()
        return docs
    
class SavingToolArgsSchema(BaseModel):
    title: str = Field(description="The title of output that represents the content well.")
    content: str = Field(description="The content you save as a txt file.")

class SavingTool(BaseTool):
    name:str = "SavingTool"
    description:str = """
        Use this tool to save given content as a txt file.
    """

    args_schema: Type[SavingToolArgsSchema] = SavingToolArgsSchema

    def _run(self, title, content):
        with open(os.path.join("files", "agent", title + ".txt"), "w",encoding="utf-8") as f:
            f.write(content)

In [19]:
agent = initialize_agent(
    llm=llm, 
    verbose=True,
    agent=AgentType.OPENAI_FUNCTIONS,
    handle_parsing_errors=True,
    tools=[
        DDGSearchTool(),
        WikiSearchTool(),
        DDGSiteLoader(),
        SavingTool()
    ],
    agent_kwargs={
        "system_message": SystemMessage(content="""
            You are a senior research engineer.
            
            You research about given topic using Wikipedia or DuckDuckGo.
            Once you find the website about the topic in DuckDuckGo, 
            extract content from the website.
            
            After full research, summarize them as good to read.
                                        
            The final output should be saved as a txt file.
        """)
    }
)

In [ ]:
query = "Research about the XZ backdoor"
result = agent.invoke(query)



> Entering new AgentExecutor chain...

Invoking: `DDGSearchTool` with `{'query': 'XZ backdoor'}`


snippet: The xz backdoor was a vulnerability in XZ Utils, a popular data compression library. The xz backdoor could let unauthorized users gain admin-level access to systems, endangering data security and much more. Read on to learn more about the xz backdoor, where it came from, and how to minimize the impact of software vulnerabilities in your systems., title: XZ Utils, the xz Backdoor & What We Can Learn from Open Source CVEs, link: https://www.puppet.com/blog/xz-backdoor, snippet: Backdooring SSH. A nefarious or compromised maintainer inserted malicious behavior in a library named liblzma, part of the xz compression tools and libraries, resulting in a backdoor in SSH.This is an advanced software supply chain attack as the library was intentionally modified for the backdoor, with obfuscation and stealth techniques for hiding the attack payload from reviewers., title: XZ Backdoor: "Th